# Data Preparation and Target Encoding

The cleaned V2 dataset was used to avoid noise from identifier variables. The data was loaded and inspected, and the target variable (reviewHelpfulness) was identified and encoded from categorical to numerical format for modelling.

In [22]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving HotelRevHelpfulnessV2.csv to HotelRevHelpfulnessV2 (2).csv


In [23]:
df = pd.read_csv("HotelRevHelpfulnessV2.csv")
print(df.shape)
df.head()

(486, 24)


,aveHelpfulnessRatioUser,stdevHelpfulnessRatioUser,pcReviewsExceedMinHelpfulnessSupport,numReviewsUser,numReviewsHotel,ratingUser,numberSubRatingsUser,subRatingMeanUser,subRatingStdevUser,aveRatingUser,...,completeness_2,completeness_3,numberTermsEntry,percentageAlphaCharsEntry,fractionUpperCaseCharsEntry,fractionYouVsIEntry,numberTermsSummaryQuote,percentageAlphaCharsSummaryQuote,fractionUpperCaseCharsSummaryQuote,reviewHelpfulness
0,1.000000,0.000000,0.666667,3,16,5,4,4.000000,0.000000,4.333333,...,0,1,182,0.788474,0.025703,0.500000,6,0.815789,0.096774,good
1,0.772487,0.377321,0.500000,12,233,5,0,0.000000,0.000000,4.333333,...,0,0,158,0.791888,0.012594,0.500000,1,1.000000,0.083333,good
2,0.715473,0.300437,0.833333,12,302,4,7,3.714286,0.755929,4.166667,...,0,3,59,0.799639,0.024831,0.333333,4,0.828571,0.034483,bad
3,0.521250,0.481675,0.222222,36,6,1,4,1.000000,0.000000,3.527778,...,0,0,95,0.782212,0.029155,0.500000,2,0.800000,0.062500,bad
4,0.603175,0.246926,1.000000,2,271,3,0,0.000000,0.000000,3.500000,...,0,0,43,0.805128,0.028662,0.000000,1,1.000000,0.142857,bad


In [24]:
df.columns

Index(['aveHelpfulnessRatioUser', 'stdevHelpfulnessRatioUser',
       'pcReviewsExceedMinHelpfulnessSupport', 'numReviewsUser',
       'numReviewsHotel', 'ratingUser', 'numberSubRatingsUser',
       'subRatingMeanUser', 'subRatingStdevUser', 'aveRatingUser',
       'stdevRatingUser', 'aveRatingHotel', 'stdevRatingHotel',
       'completeness_1', 'completeness_2', 'completeness_3',
       'numberTermsEntry', 'percentageAlphaCharsEntry',
       'fractionUpperCaseCharsEntry', 'fractionYouVsIEntry',
       'numberTermsSummaryQuote', 'percentageAlphaCharsSummaryQuote',
       'fractionUpperCaseCharsSummaryQuote', 'reviewHelpfulness'],
      dtype='object')

In [25]:
df["reviewHelpfulness"].value_counts()

,count
reviewHelpfulness,
good,308
bad,178


In [26]:
y = df["reviewHelpfulness"].map({"bad": 0, "good": 1})
X = df.drop(columns=["reviewHelpfulness"])

print(y.value_counts())

reviewHelpfulness
1    308
0    178
Name: count, dtype: int64


#✅ Q1. Naive Bayer Model Comparison

Three Naive Bayes configurations were evaluated using 10-fold cross-validation: GaussianNB, GaussianNB with feature scaling, and CategoricalNB with discretisation.

T**he GaussianNB model achieved an average accuracy of 0.654 (± 0.064)**, indicating moderate performance with relatively high variability across folds. A**pplying feature scaling did not improve performance**, with accuracy slightly decreasing to 0.648 (± 0.063), confirming that GaussianNB is largely insensitive to feature scaling as it relies on probability distributions rather than distances.

In contrast, **the CategoricalNB model with discretisation achieved the best performance, with an accuracy of 0.685 (± 0.058).** This suggests that transforming continuous variables into categorical bins improves model effectiveness for this dataset. Although discretisation may lead to information loss, in this case it improves performance by mitigating the impact of noisy continuous features and avoiding the normality assumption required by GaussianNB.

Overall, model performance is limited, with all configurations achieving only moderate accuracy. This suggests that the available features do not provide strong separation between classes, which constrains the models’ predictive ability.

**Q1.1. The Gaussian Naive Bayes model** was evaluated using 10-fold cross-validation to obtain a reliable estimate of performance on the relatively small dataset. The model achieved **an average accuracy of 0.654 with a standard deviation of 0.064.** While the mean accuracy indicates moderate predictive performance, the relatively high standard deviation suggests that the model is unstable and sensitive to variations in the training data.

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

pipeline_gnb = Pipeline([
    ('model', GaussianNB())
])

scores_gnb = cross_val_score(pipeline_gnb, X, y, cv=10)

print(f"GaussianNB mean: {scores_gnb.mean():.3f}")
print(f"GaussianNB std: {scores_gnb.std():.3f}")

GaussianNB mean: 0.654
GaussianNB std: 0.064


**Q1.2. The performance of Gaussian Naive Bayes on scaled data** does not improve after feature scaling, with a slight decrease in **accuracy from 0.654 to 0.648** and no meaningful change in variability. This indicates that scaling has little impact on GaussianNB, as the model is based on feature distributions rather than distances.


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

pipeline_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GaussianNB())
])

scores_scaled = cross_val_score(pipeline_scaled, X, y, cv=10)

print(f"GaussianNB (scaled) mean: {scores_scaled.mean():.3f}")
print(f"GaussianNB (scaled) std: {scores_scaled.std():.3f}")

GaussianNB (scaled) mean: 0.648
GaussianNB (scaled) std: 0.063


**Q1.3. The Categorical Naive Bayes** model achieved the highest performance **(accuracy = 0.685) and lower variability (std=0.058)** compared to GaussianNB. This suggests that discretising continuous features improves model performance, likely because the data does not follow the normal distribution assumption required by GaussianNB. As a result, CategoricalNB provides a better fit for this dataset. This indicates that transforming continuous variables into categorical bins reduces noise and allows the model to capture patterns more effectively.

In [29]:
import warnings
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import cross_val_score

warnings.filterwarnings("ignore")

pipeline_catnb = Pipeline([
    ('discretizer', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
    ('model', CategoricalNB())
])

scores_catnb = cross_val_score(pipeline_catnb, X, y, cv=10)

print(f"CategoricalNB mean: {scores_catnb.mean():.3f}")
print(f"CategoricalNB std: {scores_catnb.std():.3f}")

CategoricalNB mean: 0.685
CategoricalNB std: 0.058


# ✅ Q2. Decision Tree Model Selection

The Decision Tree classifier was evaluated using 10-fold cross-validation by tuning max_leaf_nodes under both Gini and Entropy criteria.

Precision (class “good”):

	•	Gini: best model at 20 leaf nodes, precision = 0.734
	•	Entropy: best model at 40 leaf nodes, precision = 0.721


Optimising for precision leads to more complex trees. **The Gini-based model achieves higher precision with fewer leaf nodes**, indicating a more efficient model for identifying the “good” class.

⸻

Accuracy Performance. Both criteria show similar performance, with Gini slightly outperforming entropy and supporting a slightly more complex optimal model. This suggests that both criteria favour relatively simple models when optimising for overall accuracy.


	•	Gini: best accuracy = 0.720 at 5 leaf nodes
	•	Entropy: best accuracy = 0.716 at 3 leaf nodes

⸻

Comparison of Accuracy and Precision

For both criteria, achieving accuracy at a level comparable to precision requires significantly fewer leaf nodes. This highlights a trade-off between overall performance and class-specific performance: accuracy favours simpler, more generalisable models, while precision leads to more complex trees focused on correctly identifying the “good” class.

⸻

Final Conclusion

The optimal model depends on the evaluation objective. When prioritising overall accuracy, both criteria favour simpler trees (Gini: 5 leaf nodes; Entropy: 3 leaf nodes). In contrast, optimising for precision on the “good” class requires more complex models (Gini: 20; Entropy: 40).

Overall, Gini achieves slightly higher performance across both metrics, while Entropy reaches comparable accuracy with a simpler tree but requires greater complexity to achieve similar precision.

**Q2.1.** When optimising for **precision on the ‘good’ class, the Decision Tree with Gini** criterion achieves the best performance at **20 leaf nodes, with a precision of 0.734.**

Precision improves with increased tree complexity up to 20 leaf nodes, then stabilises or slightly declines, suggesting diminishing returns and potential overfitting.

Overall, optimising for precision leads to selecting a larger and more specialised model focused on reducing false positives for the ‘good’ class.


In [30]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, precision_score
import pandas as pd

# precision for class "good" (1)
precision_good = make_scorer(precision_score, pos_label=1)

param_grid = {
    'max_leaf_nodes': [2, 3, 5, 7, 10, 15, 20, 30, 40, 50]
}

tree = DecisionTreeClassifier(
    criterion='gini',
    random_state=42
)

gs_precision = GridSearchCV(
    tree,
    param_grid,
    cv=10,
    scoring=precision_good,
    n_jobs=-1
)

gs_precision.fit(X, y)

print("Best params (precision):", gs_precision.best_params_)
print("Best precision:", round(gs_precision.best_score_, 3))

results_precision = pd.DataFrame(gs_precision.cv_results_)[
    ['param_max_leaf_nodes', 'mean_test_score', 'std_test_score']
].sort_values('param_max_leaf_nodes')

results_precision = results_precision.rename(columns={
    'param_max_leaf_nodes': 'max_leaf_nodes',
    'mean_test_score': 'mean_precision',
    'std_test_score': 'std'
}).round(3)

results_precision

Best params (precision): {'max_leaf_nodes': 20}
Best precision: 0.734


,max_leaf_nodes,mean_precision,std
0,2,0.692,0.034
1,3,0.700,0.033
2,5,0.731,0.033
3,7,0.727,0.031
4,10,0.726,0.031
5,15,0.725,0.039
6,20,0.734,0.037
7,30,0.724,0.037
8,40,0.727,0.021
9,50,0.723,0.030


**Q2.2.** The optimal Decision Tree under the **entropy** criterion, when optimising precision **for the “good” class**, is obtained at **40 leaf nodes**, achieving a **precision of 0.721.**

In [31]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, precision_score
import pandas as pd

# precision for class "good" (encoded as 1)
precision_good = make_scorer(precision_score, pos_label=1)

# parameter grid
param_grid_entropy = {
    'max_leaf_nodes': [2, 3, 5, 7, 10, 15, 20, 30, 40, 50]
}

# model (Entropy criterion, fixed randomness)
tree_entropy = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)

# grid search with 10-fold CV, optimising precision for the "good" class
gs_entropy_precision = GridSearchCV(
    tree_entropy,
    param_grid_entropy,
    cv=10,
    scoring=precision_good,
    n_jobs=-1
)

# run search
gs_entropy_precision.fit(X, y)

# best result
print("Best entropy params (precision):", gs_entropy_precision.best_params_)
print("Best entropy precision:", round(gs_entropy_precision.best_score_, 3))

# all results
results_entropy_precision = pd.DataFrame(gs_entropy_precision.cv_results_)[
    ['param_max_leaf_nodes', 'mean_test_score', 'std_test_score']
].sort_values('param_max_leaf_nodes')

results_entropy_precision = results_entropy_precision.rename(columns={
    'param_max_leaf_nodes': 'max_leaf_nodes',
    'mean_test_score': 'mean_precision',
    'std_test_score': 'std'
}).round(3)

results_entropy_precision

Best entropy params (precision): {'max_leaf_nodes': 40}
Best entropy precision: 0.721


,max_leaf_nodes,mean_precision,std
0,2,0.692,0.034
1,3,0.700,0.033
2,5,0.712,0.036
3,7,0.715,0.035
4,10,0.716,0.036
5,15,0.709,0.042
6,20,0.711,0.050
7,30,0.711,0.051
8,40,0.721,0.049
9,50,0.716,0.051


**Q2. Additional:** Accuracy Performance Comparison

The Decision Tree (Gini) achieves its highest accuracy of 0.720 at 5 leaf nodes, indicating that a relatively simple model provides the best overall classification performance.

In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# parameter grid
param_grid_gini = {
    'max_leaf_nodes': [2, 3, 5, 7, 10, 15, 20, 30, 40, 50]
}

# model (Gini criterion, fixed randomness)
tree_gini = DecisionTreeClassifier(
    criterion='gini',
    random_state=42
)

# grid search with 10-fold CV
gs_gini = GridSearchCV(
    tree_gini,
    param_grid_gini,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

# run search
gs_gini.fit(X, y)

# best result
print("Best gini params:", gs_gini.best_params_)
print("Best gini accuracy:", round(gs_gini.best_score_, 3))

# all results
results_gini = pd.DataFrame(gs_gini.cv_results_)[
    ['param_max_leaf_nodes', 'mean_test_score', 'std_test_score']
].sort_values('param_max_leaf_nodes')

results_gini = results_gini.rename(columns={
    'param_max_leaf_nodes': 'max_leaf_nodes',
    'mean_test_score': 'mean_accuracy',
    'std_test_score': 'std'
}).round(3)

results_gini

Best gini params: {'max_leaf_nodes': 5}
Best gini accuracy: 0.72


,max_leaf_nodes,mean_accuracy,std
0,2,0.679,0.063
1,3,0.716,0.051
2,5,0.720,0.050
3,7,0.702,0.041
4,10,0.687,0.036
5,15,0.679,0.068
6,20,0.689,0.070
7,30,0.673,0.082
8,40,0.675,0.051
9,50,0.670,0.061


The Decision Tree (entropy) achieves its highest accuracy of 0.716 at 3 leaf nodes, indicating that optimal performance is reached with a simpler model compared to the Gini-based tree.


In [33]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

# parameter grid
param_grid_entropy = {
    'max_leaf_nodes': [2, 3, 5, 7, 10, 15, 20, 30, 40, 50]
}

# model (Entropy criterion, fixed randomness)
tree_entropy = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)

# grid search with 10-fold CV
gs_entropy = GridSearchCV(
    tree_entropy,
    param_grid_entropy,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

# run search
gs_entropy.fit(X, y)

# best result
print("Best entropy params:", gs_entropy.best_params_)
print("Best entropy accuracy:", round(gs_entropy.best_score_, 3))

# all results
results_entropy = pd.DataFrame(gs_entropy.cv_results_)[
    ['param_max_leaf_nodes', 'mean_test_score', 'std_test_score']
].sort_values('param_max_leaf_nodes')

results_entropy = results_entropy.rename(columns={
    'param_max_leaf_nodes': 'max_leaf_nodes',
    'mean_test_score': 'mean_accuracy',
    'std_test_score': 'std'
}).round(3)

results_entropy

Best entropy params: {'max_leaf_nodes': 3}
Best entropy accuracy: 0.716


,max_leaf_nodes,mean_accuracy,std
0,2,0.679,0.063
1,3,0.716,0.051
2,5,0.710,0.052
3,7,0.683,0.043
4,10,0.671,0.075
5,15,0.665,0.073
6,20,0.658,0.081
7,30,0.638,0.086
8,40,0.650,0.074
9,50,0.646,0.075


# ✅ Q3. Pipeline Execution and Data Flow



**Q3.1. When kNNpipe.fit(X_train, y_train)** is executed, the pipeline sequentially applies all preprocessing steps followed by model training using the training data.

• First, KNNImputer applies fit and transform on X_train to learn how to impute missing values and perform the imputation, where missing values are estimated based on the nearest neighbours using feature similarity.

• Next, StandardScaler applies fit and transform, ensuring that all variables are on a comparable scale so that no feature dominates the distance calculations.

• Finally, KNeighborsClassifier performs fit using the transformed training data and y_train.

All operations are performed on the training data only, ensuring proper model learning without data leakage.
___

**X_train → Imputer (fit+transform) → Scaler (fit+transform) → Classifier (fit with y_train)**
___


Additional explanation

KNNImputer replaces missing values using nearest neighbours in the feature space, enabling data-driven imputation that maintains local structure and reduces distortion.

StandardScaler ensures that all features contribute equally to distance calculations, which is essential for distance-based models such as k-NN.

**Q3.2. In line 6, y_pred = kNNpipe.predict(X_test)**, the method predict is called on the Pipeline object, with X_test passed as input.

Inside the pipeline, the following methods are executed:

• KNNImputer.transform(X_test) — imputes missing values using parameters learned from X_train
• StandardScaler.transform(X_test) — scales the data using parameters learned from X_train
• KNeighborsClassifier.predict(X_test) — generates predictions from the transformed data

The final output of kNNpipe.predict(X_test) is the predicted labels, which are assigned to y_pred.

No fit methods are called at this stage.
___
**X_test → Imputer (transform) → Scaler (transform) → Classifier (predict) → y_pred**

# ✅ Q4. Severity of “cheating”, where 0 – harmless & 4 - very serious

**a) Mean imputation before split → 2**

Mean imputation uses summary statistics (mean) computed from the entire dataset, including the test set. This introduces data leakage, but the effect is relatively limited as it does not use relationships between observations.  

**b) k-NN imputation before split → 4**

k-NN imputation uses feature similarity across all observations, including the test data. This introduces strong data leakage because the imputed values depend on relationships between training and test examples.

**c) Cyclical encoding → 0**

Cyclical encoding is a deterministic transformation (e.g. sin/cos) that does not depend on the data distribution. It does not use any information from the dataset, so it introduces no leakage.

**d) Feature selection before split → 4**

Feature selection uses the relationship between features and the target variable to decide which features are important. If this is done before the split, these relationships are learned using both training and test data. As a result, the selected features are biased towards patterns that exist in the test set, making the model evaluation unreliable.

**e) Model trained before split → 4**

Training the model before splitting means the model is fitted on the entire dataset, including the test data. As a result, the model has already learned patterns from the test set, leading to overly optimistic evaluation and effectively overfitting to the data it is later evaluated on.

**f) Scaling before split → 2**

Scaling uses statistics such as mean and variance computed from the entire dataset. This introduces leakage, but it is weaker than methods that exploit relationships or target information.

**g) Hyperparameters determined before split → 4**

Hyperparameter tuning selects the best model settings by evaluating performance. If this is done before the split, the evaluation is based on the full dataset, including the test data. As a result, the chosen hyperparameters are those that perform best not only on the training data but also on the test set. This means the model is indirectly tuned to the test data, leading to overly optimistic performance.

**h) SMOTE before split → 4**

SMOTE (Synthetic Minority Over-sampling Technique) uses relationships between observations to generate synthetic samples. If applied before splitting, these relationships include the test data, causing the training data to reflect the structure of the test set and leading to strong data leakage.

# ✅ Q5a. Number of Models in Grid Search

The total number of unique models evaluated is determined by the number of parameter combinations in the grid.

Parameters:

	•	n_neighbors: 4 options → [1, 3, 5, 10]
	•	metric: 3 options → [‘manhattan’, ‘euclidean’, ‘cosine’]
	•	weights: 2 options → [‘uniform’, ‘distance’]

The total number of combinations is: 4 × 3 × 2 = 24

Therefore, **24 unique nearest neighbour models are evaluated**.

Each of these models is then assessed using 10-fold cross-validation.


Randomised search samples parameter combinations randomly rather than exhaustively exploring all possible combinations. When some parameters have little or no effect on model performance, exhaustive grid search wastes computational effort evaluating all their possible values. In contrast, randomised search allocates more trials across the full parameter space, increasing the chance of identifying good values for the important parameters while not over-exploring irrelevant ones. This makes it less sensitive to parameters that do not significantly affect model performance.

# ✅ Q5b. Randomised vs Exhaustive Grid Search

Randomised search samples parameter combinations rather than exhaustively evaluating all possible combinations as in grid search. When some parameters have little impact on model performance, exhaustive grid search wastes evaluations exploring irrelevant dimensions. In contrast, **randomised search allocates evaluations more efficiently across the parameter space, increasing the likelihood of identifying good values for the most important parameters** under a fixed computational budget.


# Convert ipynb to HTML in Colab

In [35]:
#@title Convert ipynb to HTML in Colab
# Upload ipynb
from google.colab import files
f = files.upload()

# Convert ipynb to html
import subprocess
file0 = list(f.keys())[0]
_ = subprocess.run(["pip", "install", "nbconvert"])
_ = subprocess.run(["jupyter", "nbconvert", file0, "--to", "html"])

# download the html
files.download(file0[:-5]+"html")

Saving COMP47990 - Submission - Shvoriak - 25255818 - Assigment 2.ipynb to COMP47990 - Submission - Shvoriak - 25255818 - Assigment 2.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>